In [1]:
import sys
import subprocess
import numpy as np
from pathlib import Path

try:
    import plotly.graph_objects as go
except ModuleNotFoundError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "plotly"])
    import plotly.graph_objects as go

rng = np.random.default_rng(42)
FIG_DIR = Path("figures_case3")
FIG_DIR.mkdir(exist_ok=True)

## 1. Данные

In [2]:
def distance_matrix(Y):
    diff = Y[:, None, :] - Y[None, :, :]
    return np.sqrt(np.sum(diff ** 2, axis=2))

n = 32
Y_true = rng.normal(size=(n, 3))
D = distance_matrix(Y_true)

print("n =", n)
print("размерность исходных точек =", Y_true.shape[1])
print("размер D =", D.shape)
print("первые 5x5 элементов D:")
print(np.round(D[:5, :5], 3))

n = 32
размерность исходных точек = 3
размер D = (32, 32)
первые 5x5 элементов D:
[[0.    2.334 1.069 2.242 2.199]
 [2.334 0.    2.233 3.944 3.657]
 [1.069 2.233 0.    1.739 1.524]
 [2.242 3.944 1.739 0.    1.001]
 [2.199 3.657 1.524 1.001 0.   ]]


## 2. Проверка матрицы расстояний

In [3]:
def check_distance_matrix(D, sample=None, seed=0, tol=1e-10):
    n = D.shape[0]
    diag_error = float(np.max(np.abs(np.diag(D))))
    symmetry_error = float(np.max(np.abs(D - D.T)))
    min_value = float(np.min(D))
    triples = [(i, j, k) for i in range(n) for j in range(n) for k in range(n)]
    if sample is not None and sample < len(triples):
        r = np.random.default_rng(seed)
        idx = r.choice(len(triples), size=sample, replace=False)
        triples = [triples[t] for t in idx]
    violations = 0
    worst_gap = 0.0
    for i, j, k in triples:
        gap = D[i, k] - D[i, j] - D[j, k]
        if gap > tol:
            violations += 1
            worst_gap = max(worst_gap, float(gap))
    return {
        "diag_error": diag_error,
        "symmetry_error": symmetry_error,
        "min_value": min_value,
        "triangle_checked": len(triples),
        "triangle_violations": violations,
        "triangle_violation_share": violations / len(triples),
        "worst_triangle_gap": worst_gap,
    }

checks = check_distance_matrix(D)
for key, value in checks.items():
    print(key, ":", value)

diag_error : 0.0
symmetry_error : 0.0
min_value : 0.0
triangle_checked : 32768
triangle_violations : 0
triangle_violation_share : 0.0
worst_triangle_gap : 0.0


## 3. Двойное центрирование

$$d_{ij}^2=b_{ii}+b_{jj}-2b_{ij}$$

$$J=I-\frac1n\mathbf 1\mathbf 1^T, \qquad B=-\frac12JD^{(2)}J$$

Центрирование фиксирует перенос: после него сумма координат равна нулю.

In [4]:
def gram_from_distances(D):
    n = D.shape[0]
    D2 = D ** 2
    J = np.eye(n) - np.ones((n, n)) / n
    B = -0.5 * J @ D2 @ J
    return (B + B.T) / 2, J, D2

B, J, D2 = gram_from_distances(D)

print("||B - B.T||_max =", np.max(np.abs(B - B.T)))
print("||J1||_max =", np.max(np.abs(J @ np.ones(n))))
print("||B1||_max =", np.max(np.abs(B @ np.ones(n))))
print("первые 5x5 элементов B:")
print(np.round(B[:5, :5], 3))

||B - B.T||_max = 0.0
||J1||_max = 0.0
||B1||_max = 8.881784197001252e-15
первые 5x5 элементов B:
[[ 1.425  0.94   0.163 -0.614 -0.816]
 [ 0.94   5.902  0.481 -3.641 -2.847]
 [ 0.163  0.481  0.045 -0.303 -0.25 ]
 [-0.614 -3.641 -0.303  2.372  1.573]
 [-0.816 -2.847 -0.25   1.573  1.776]]


## 4. Спектр матрицы Грама

Для симметричной матрицы используется `np.linalg.eigh`: собственные значения вещественные, а собственные векторы ортонормированы.

In [5]:
def spectrum(B):
    vals, vecs = np.linalg.eigh(B)
    idx = np.argsort(vals)[::-1]
    return vals[idx], vecs[:, idx]

eigvals, eigvecs = spectrum(B)
tol = 1e-10 * max(1.0, float(np.max(np.abs(eigvals))))
positive = int(np.sum(eigvals > tol))
zero = int(np.sum(np.abs(eigvals) <= tol))
negative = int(np.sum(eigvals < -tol))

print("tol =", tol)
print("положительных =", positive)
print("нулевых =", zero)
print("отрицательных =", negative)
print("ранг B =", positive)
print("первые собственные значения:")
print(np.round(eigvals[:10], 10))

tol = 2.9660032087983257e-09
положительных = 3
нулевых = 29
отрицательных = 0
ранг B = 3
первые собственные значения:
[29.66003209 18.79241177  7.19447121  0.          0.          0.
  0.          0.          0.          0.        ]


## 5. Восстановление координат

$$B = UΛ U^T, \quad Y = U_r Λ_r^{1/2}$$

$$YY^T = U_r Λ_r U_r^T = B_r$$

In [6]:
def coords_from_spectrum(vals, vecs, m=None, tol=None):
    if tol is None:
        tol = 1e-10 * max(1.0, float(np.max(np.abs(vals))))
    pos = vals > tol
    vals_pos = vals[pos]
    vecs_pos = vecs[:, pos]
    if m is not None:
        vals_pos = vals_pos[:m]
        vecs_pos = vecs_pos[:, :m]
    return vecs_pos * np.sqrt(vals_pos)

def reconstruction_errors(D, Y):
    D_hat = distance_matrix(Y)
    E = D - D_hat
    return {
        "E_max": float(np.max(np.abs(E))),
        "E_F": float(np.linalg.norm(E, "fro")),
        "E_rel": float(np.linalg.norm(E, "fro") / np.linalg.norm(D, "fro")),
    }, D_hat

Y_full = coords_from_spectrum(eigvals, eigvecs)
Y2 = coords_from_spectrum(eigvals, eigvecs, 2)
Y3 = coords_from_spectrum(eigvals, eigvecs, 3)

for name, Y in [("full", Y_full), ("2D", Y2), ("3D", Y3)]:
    err, _ = reconstruction_errors(D, Y)
    print(name, err)

full {'E_max': 2.6645352591003757e-15, 'E_F': 2.837222673621981e-14, 'E_rel': 4.754259558488426e-16}
2D {'E_max': 1.4903451444595581, 'E_F': 8.4370703601074, 'E_rel': 0.141377773334841}
3D {'E_max': 2.6645352591003757e-15, 'E_F': 2.837222673621981e-14, 'E_rel': 4.754259558488426e-16}


In [7]:
def procrustes_error(A, B):
    A0 = A - A.mean(axis=0)
    B0 = B - B.mean(axis=0)
    M = B0.T @ A0
    U, _, Vt = np.linalg.svd(M)
    R = U @ Vt
    B_aligned = B0 @ R
    return float(np.sqrt(np.mean(np.sum((A0 - B_aligned) ** 2, axis=1))))

print("ошибка Прокруста для 3D =", procrustes_error(Y_true, Y3))

ошибка Прокруста для 3D = 9.234533254834286e-16


## 6. Графики

In [8]:
def save_plotly(fig, name):
    html_path = FIG_DIR / f"{name}.html"
    fig.write_html(html_path, include_plotlyjs="cdn")
    try:
        fig.write_image(FIG_DIR / f"{name}.png", scale=2)
    except Exception:
        pass
    return fig

plotly_layout = {
    "template": "plotly_white",
    "font": {"family": "Arial, sans-serif", "size": 14},
    "margin": {"l": 60, "r": 30, "t": 70, "b": 55},
}

In [9]:
fig = go.Figure()
fig.add_trace(go.Scatter(x=np.arange(1, len(eigvals) + 1), y=eigvals, mode="lines+markers", name="lambda"))
fig.update_layout(**plotly_layout, title="Собственные значения B", xaxis_title="номер", yaxis_title="значение")
save_plotly(fig, "eigenvalues")

In [10]:
fig = go.Figure()
fig.add_trace(go.Scatter(x=Y2[:, 0], y=Y2[:, 1], mode="markers+text", text=[str(i) for i in range(n)], textposition="top center", name="точки"))
fig.update_layout(**plotly_layout, title="2D-вложение", xaxis_title="y1", yaxis_title="y2")
fig.update_yaxes(scaleanchor="x", scaleratio=1)
save_plotly(fig, "embedding_2d")

In [11]:
fig = go.Figure()
fig.add_trace(go.Scatter3d(x=Y3[:, 0], y=Y3[:, 1], z=Y3[:, 2], mode="markers+text", text=[str(i) for i in range(n)], marker={"size": 5}, name="точки"))
fig.update_layout(**plotly_layout, title="3D-вложение", scene={"xaxis_title": "y1", "yaxis_title": "y2", "zaxis_title": "y3", "aspectmode": "data"})
save_plotly(fig, "embedding_3d")

## 7. Зависимость от размерности

In [12]:
m_grid = list(range(1, positive + 1))
errors_by_m = []
rel_by_m = []

for m in m_grid:
    err, _ = reconstruction_errors(D, coords_from_spectrum(eigvals, eigvecs, m))
    errors_by_m.append(err["E_F"])
    rel_by_m.append(err["E_rel"])
    print(m, "E_F =", err["E_F"], "E_rel =", err["E_rel"])

fig = go.Figure()
fig.add_trace(go.Scatter(x=m_grid, y=errors_by_m, mode="lines+markers", name="E_F"))
fig.add_trace(go.Scatter(x=m_grid, y=rel_by_m, mode="lines+markers", name="E_rel", yaxis="y2"))
fig.update_layout(**plotly_layout, title="Ошибка восстановления от размерности", xaxis_title="m", yaxis={"title": "E_F"}, yaxis2={"title": "E_rel", "overlaying": "y", "side": "right"})
save_plotly(fig, "error_by_dimension")

1 E_F = 25.33885453135039 E_rel = 0.4245965340571324
2 E_F = 8.4370703601074 E_rel = 0.141377773334841
3 E_F = 2.837222673621981e-14 E_rel = 4.754259558488426e-16


## 8. Устойчивость к шуму

In [13]:
eps_grid = np.array([0, 0.01, 0.03, 0.05, 0.10, 0.15, 0.20, 0.30])
base_noise = rng.normal(size=(n, n))
base_noise = (base_noise + base_noise.T) / 2
np.fill_diagonal(base_noise, 0)

noise_negative = []
noise_errors = []
noise_triangle = []
noise_spectra = []

for eps in eps_grid:
    Dn = D + eps * base_noise
    Dn = np.maximum(Dn, 0)
    Dn = (Dn + Dn.T) / 2
    np.fill_diagonal(Dn, 0)
    Bn, _, _ = gram_from_distances(Dn)
    vals_n, vecs_n = spectrum(Bn)
    tol_n = 1e-10 * max(1.0, float(np.max(np.abs(vals_n))))
    Yn = coords_from_spectrum(vals_n, vecs_n, min(3, int(np.sum(vals_n > tol_n))), tol_n)
    err_n, _ = reconstruction_errors(Dn, Yn)
    checks_n = check_distance_matrix(Dn)
    noise_negative.append(int(np.sum(vals_n < -tol_n)))
    noise_errors.append(err_n["E_rel"])
    noise_triangle.append(checks_n["triangle_violation_share"])
    noise_spectra.append(vals_n[:8])
    print(eps, noise_negative[-1], noise_errors[-1], noise_triangle[-1])

0.0 0 4.754259558488426e-16 0.0
0.01 14 0.003715702448710498 0.00177001953125
0.03 14 0.011172281088427086 0.005126953125
0.05 14 0.01866109733794486 0.00860595703125
0.1 14 0.037513068105168315 0.01519775390625
0.15 14 0.0565274668413219 0.02020263671875
0.2 14 0.07567253786359744 0.0274658203125
0.3 14 0.1140904346857463 0.040283203125


In [14]:
fig = go.Figure()
fig.add_trace(go.Scatter(x=eps_grid, y=noise_errors, mode="lines+markers", name="E_rel"))
fig.add_trace(go.Scatter(x=eps_grid, y=noise_triangle, mode="lines+markers", name="доля нарушений"))
fig.update_layout(**plotly_layout, title="Шум: ошибки и нарушения", xaxis_title="epsilon", yaxis_title="значение")
save_plotly(fig, "noise_errors")

In [15]:
fig = go.Figure()
fig.add_trace(go.Scatter(x=eps_grid, y=noise_negative, mode="lines+markers", name="отрицательные"))
fig.update_layout(**plotly_layout, title="Шум: отрицательные собственные значения", xaxis_title="epsilon", yaxis_title="количество")
save_plotly(fig, "noise_negative")

In [16]:
noise_spectra = np.array(noise_spectra)
fig = go.Figure()
for i in range(5):
    fig.add_trace(go.Scatter(x=eps_grid, y=noise_spectra[:, i], mode="lines+markers", name=f"lambda_{i + 1}"))
fig.update_layout(**plotly_layout, title="Шум: первые собственные значения", xaxis_title="epsilon", yaxis_title="значение")
save_plotly(fig, "noise_spectrum")

## 9. Изометричность и визуализация

In [17]:
triu = np.triu_indices(n, 1)
_, D_2 = reconstruction_errors(D, Y2)
corr_2 = float(np.corrcoef(D[triu], D_2[triu])[0, 1])
lo = min(D[triu].min(), D_2[triu].min())
hi = max(D[triu].max(), D_2[triu].max())

print("corr(D, D_hat_2D) =", corr_2)
fig = go.Figure()
fig.add_trace(go.Scatter(x=D[triu], y=D_2[triu], mode="markers", name="пары"))
fig.add_trace(go.Scatter(x=[lo, hi], y=[lo, hi], mode="lines", name="идеал"))
fig.update_layout(**plotly_layout, title=f"Исходные и восстановленные расстояния, corr={corr_2:.4f}", xaxis_title="d_ij", yaxis_title="d_hat_ij")
save_plotly(fig, "distance_scatter")

corr(D, D_hat_2D) = 0.9627467145753552


## 10. Точки на окружности

In [18]:
theta = np.linspace(0, 2*np.pi, 40, endpoint=False)
Y_circle = np.c_[np.cos(theta), np.sin(theta)]
D_circle = distance_matrix(Y_circle)
B_circle, _, _ = gram_from_distances(D_circle)
vals_c, vecs_c = spectrum(B_circle)
Y_circle_rec = coords_from_spectrum(vals_c, vecs_c, 2)
err_circle, _ = reconstruction_errors(D_circle, Y_circle_rec)

print("первые собственные значения:", np.round(vals_c[:6], 10))
print("ошибки:", err_circle)
fig = go.Figure()
fig.add_trace(go.Scatter(x=Y_circle_rec[:, 0], y=Y_circle_rec[:, 1], mode="markers+lines", name="точки"))
fig.update_layout(**plotly_layout, title="Точки окружности после восстановления", xaxis_title="y1", yaxis_title="y2")
fig.update_yaxes(scaleanchor="x", scaleratio=1)
save_plotly(fig, "circle_embedding")

первые собственные значения: [20. 20.  0.  0.  0.  0.]
ошибки: {'E_max': 1.9984014443252818e-15, 'E_F': 1.9592684222110886e-14, 'E_rel': 3.463529968775321e-16}


## 11. Реальные данные: города

In [19]:
import pandas as pd

def load_worldcities():
    usecols = ["Country", "AccentCity", "Population", "Latitude", "Longitude"]
    chunks = pd.read_csv("worldcitiespop.csv", usecols=usecols, chunksize=250_000)
    parts = []
    for ch in chunks:
        ru = ch[(ch["Country"].astype(str).str.lower() == "ru") & ch["Population"].notna()]
        if len(ru):
            parts.append(ru)
    cities = pd.concat(parts, ignore_index=True)
    cities = cities.sort_values("Population", ascending=False)
    cities = cities.drop_duplicates(subset=["AccentCity"])
    return cities.head(40).reset_index(drop=True)

cities = load_worldcities()
lat = np.deg2rad(cities["Latitude"].to_numpy(float))
lon = np.deg2rad(cities["Longitude"].to_numpy(float))
lat0 = lat.mean()
earth_radius = 6371.0
X_geo = np.column_stack([earth_radius * np.cos(lat0) * lon, earth_radius * lat])
D_geo = distance_matrix(X_geo)
B_geo = gram_from_distances(D_geo)[0] if isinstance(gram_from_distances(D_geo), tuple) else gram_from_distances(D_geo)
eig_geo, vec_geo = spectrum(B_geo)
Y_geo_2 = coords_from_spectrum(eig_geo, vec_geo, 2)
err_geo, D_geo_hat = reconstruction_errors(D_geo, Y_geo_2)
Y_geo_aligned, X_geo_centered = procrustes_align(X_geo, Y_geo_2)
triu = np.triu_indices(len(cities), 1)
corr_geo = float(np.corrcoef(D_geo[triu], D_geo_hat[triu])[0, 1])
tol_geo = 1e-10 * max(1.0, float(np.max(np.abs(eig_geo))))

print("число городов =", len(cities))
print("положительных собственных значений =", int(np.sum(eig_geo > tol_geo)))
print("отрицательных собственных значений =", int(np.sum(eig_geo < -tol_geo)))
print("ошибки восстановления =", err_geo)
print("corr(D, D_hat) =", corr_geo)
print(cities[["AccentCity", "Population", "Latitude", "Longitude"]].head(10).to_string(index=False))

число городов = 40
положительных собственных значений = 2
отрицательных собственных значений = 0
ошибки восстановления = {'E_max': 1.0913936421275139e-11, 'E_F': 8.556165270563296e-11, 'E_rel': 8.670922845097687e-16}
corr(D, D_hat) = 1.0
      AccentCity  Population  Latitude  Longitude
          Moscow  10381288.0 55.752222  37.615556
Saint Petersburg   4039751.0 59.894444  30.264167
     Novosibirsk   1419016.0 55.041500  82.934600
   Yekaterinburg   1287586.0 56.851900  60.612200
Nizhniy Novgorod   1284176.0 56.326944  44.007500
          Samara   1134742.0 53.183500  50.118200
            Omsk   1129289.0 55.000000  73.400000
           Kazan   1104750.0 55.788740  49.122144
  Rostov-na-Donu   1074495.0 47.231350  39.723284
     Chelyabinsk   1062931.0 55.154444  61.429722


In [20]:
fig = go.Figure()
fig.add_trace(go.Scatter(x=X_geo_centered[:, 0], y=X_geo_centered[:, 1], mode="markers+text", text=cities["AccentCity"], textposition="top center", name="реальные координаты"))
fig.update_layout(**plotly_layout, title="Города России: координаты в плоской проекции", xaxis_title="x, км", yaxis_title="y, км")
fig.update_yaxes(scaleanchor="x", scaleratio=1)
save_plotly(fig, "worldcities_real")

In [21]:
fig = go.Figure()
fig.add_trace(go.Scatter(x=Y_geo_aligned[:, 0], y=Y_geo_aligned[:, 1], mode="markers+text", text=cities["AccentCity"], textposition="top center", name="MDS"))
fig.update_layout(**plotly_layout, title="Города России: восстановление по расстояниям", xaxis_title="x, км", yaxis_title="y, км")
fig.update_yaxes(scaleanchor="x", scaleratio=1)
save_plotly(fig, "worldcities_mds")

In [22]:
fig = go.Figure()
fig.add_trace(go.Scatter(x=np.arange(1, len(eig_geo) + 1), y=eig_geo, mode="lines+markers", name="lambda"))
fig.update_layout(**plotly_layout, title="Города России: спектр матрицы Грама", xaxis_title="номер", yaxis_title="значение")
save_plotly(fig, "worldcities_spectrum")

In [23]:
lo = min(D_geo[triu].min(), D_geo_hat[triu].min())
hi = max(D_geo[triu].max(), D_geo_hat[triu].max())
fig = go.Figure()
fig.add_trace(go.Scatter(x=D_geo[triu], y=D_geo_hat[triu], mode="markers", name="пары"))
fig.add_trace(go.Scatter(x=[lo, hi], y=[lo, hi], mode="lines", name="идеал"))
fig.update_layout(**plotly_layout, title=f"Города России: расстояния, corr={corr_geo:.4f}", xaxis_title="исходное расстояние", yaxis_title="восстановленное расстояние")
save_plotly(fig, "worldcities_distances")

## 11. Выводы

1. Для точной евклидовой матрицы расстояний матрица Грама положительно полуопределена.
2. Ранг `B` равен минимальной размерности точного вложения.
3. При усечении спектра ошибка расстояний растет.
4. Шум приводит к отрицательным собственным значениям и нарушениям метрических свойств.
5. Хорошая 2D-картинка не гарантирует точного сохранения всех расстояний.